In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

# for NLP tasks
import torch
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq
from datasets import Dataset
import evaluate

/home/dangtrxn/cs4742_nlp/AmazonReview_Project/Amazon_Review/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# load dataset
root_dir = Path().resolve().parent
data_path = root_dir / 'data' / 'refined_summaries.csv'
data = pd.read_csv(data_path, delimiter=',')

# only keep Summary, Text columns
data = data[['Summary','Text']]

# rename columns
data.rename(columns={"Summary": "summary", "Text": "text"}, inplace=True)

# drop empty rows
data = data.dropna()

# display info and head
data.info()
data.head()

<class 'pandas.DataFrame'>
RangeIndex: 1267 entries, 0 to 1266
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   summary  1267 non-null   str  
 1   text     1267 non-null   str  
dtypes: str(2)
memory usage: 637.7 KB


,summary,text
0,High-quality dog food with stew-like texture a...,I have bought several of the Vitality canned d...
1,Misleading product labeling; received small un...,Product arrived labeled as Jumbo Salted Peanut...
2,"Light, flavorful Turkish delight with pleasant...",This is a confection that has been around a fe...
3,Cherry soda flavor tastes overly medicinal and...,If you are looking for the secret ingredient i...
4,"Affordable, tasty taffy with good variety and ...",Great taffy at a great price. There was a wid...


In [4]:
# split dataset
train, temp = train_test_split(data, test_size=.2, random_state=42)
val, test = train_test_split(temp, test_size=.5, random_state=42)

# convert from Pandas dataframe to HuggingFace Dataset format
train_dataset = Dataset.from_pandas(train.reset_index(drop=True))
val_dataset = Dataset.from_pandas(val.reset_index(drop=True))
test_dataset = Dataset.from_pandas(test.reset_index(drop=True))

In [ ]:
model_name = 'facebook/bart-large-cnn'

tokenizer = AutoTokenizer.from_pretrained(model_name)

max_input = 512

# tokenizer function
def tokenize(batch):
    model_inputs = tokenizer(
        batch['text'],
        max_length = max_input,
        padding = 'max_length',
        truncation = True
    )

    labels = tokenizer(
        batch['summary'],
        max_length = max_input,
        padding = 'max_length',
        truncation = True
    )

    model_inputs['labels'] = labels['input_ids']
    return model_inputs

# apply tokenize to each dataset
tokenized_train = train_dataset.map(tokenize, batched=True, remove_columns=['text','summary'])
tokenized_val = val_dataset.map(tokenize, batched=True, remove_columns=['text','summary'])
tokenized_test = test_dataset.map(tokenize, batched=True, remove_columns=['text','summary'])

Map: 100%|██████████| 127/127 [00:00<00:00, 3644.18 examples/s]


In [6]:
rouge = evaluate.load('rouge')

# function to compute rouge scores while training
def compute_rouge(pred):
    predictions, labels = pred
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    # replace -100 padding tokens in labels before decoding
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    result = rouge.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=True)
    return {k: round(v * 100, 4) for k, v in result.items()}

In [8]:
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

args = Seq2SeqTrainingArguments(
    output_dir='bart-finetuned-reviews-v3',
    eval_strategy='epoch',
    save_strategy='epoch',
    learning_rate=2e-5,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    weight_decay=0.01,
    num_train_epochs=10,
    predict_with_generate=True,
    fp16=torch.cuda.is_available(),
    save_total_limit=2,  
    load_best_model_at_end=True,
    logging_steps=500
)

collator = DataCollatorForSeq2Seq(tokenizer, model=model, padding=True)

trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    tokenizer=tokenizer,
    data_collator=collator,
    compute_metrics=compute_rouge
)

/tmp/ipykernel_345489/435464623.py:22: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


In [9]:
trainer.train()

/home/dangtrxn/cs4742_nlp/AmazonReview_Project/Amazon_Review/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel,Rougelsum
0,No log,0.097283,1.098800,0.242500,0.889300,0.881700
1,No log,0.085790,3.856400,0.573800,3.129800,3.117200
2,No log,0.089328,5.930800,0.948000,4.761000,4.774000
3,0.495500,0.095542,6.054400,0.763300,4.949200,4.984700
4,0.495500,0.107500,6.448400,0.800100,5.395800,5.385200
5,0.495500,0.116828,6.136200,0.945600,5.083100,5.096000
6,0.495500,0.118143,6.894800,1.020100,5.747800,5.755200
7,0.014100,0.127414,7.826600,1.037300,6.522000,6.540500
8,0.014100,0.127756,7.304200,0.926600,6.182700,6.177000
9,0.014100,0.127802,7.294900,0.782300,5.858200,5.853700


/home/dangtrxn/cs4742_nlp/AmazonReview_Project/Amazon_Review/lib/python3.12/site-packages/transformers/modeling_utils.py:2810: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 142, 'min_length': 56, 'early_stopping': True, 'num_beams': 4, 'length_penalty': 2.0, 'no_repeat_ngram_size': 3, 'forced_bos_token_id': 0}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(
/home/dangtrxn/cs4742_nlp/AmazonReview_Project/Amazon_Review/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/home/dangtrxn/cs4742_nlp/AmazonReview_Project/Amazon_Review/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then dev

TrainOutput(global_step=1260, training_loss=0.20342195336780852, metrics={'train_runtime': 48802.7143, 'train_samples_per_second': 0.208, 'train_steps_per_second': 0.026, 'total_flos': 1.0941711136063488e+16, 'train_loss': 0.20342195336780852, 'epoch': 9.994082840236686})

In [10]:
# save fine-tuned model
trainer.save_model('bart-finetuned-reviews-v3')
tokenizer.save_pretrained('bart-finetuned-reviews-v3')

('bart-finetuned-reviews-v3/tokenizer_config.json',
 'bart-finetuned-reviews-v3/special_tokens_map.json',
 'bart-finetuned-reviews-v3/vocab.json',
 'bart-finetuned-reviews-v3/merges.txt',
 'bart-finetuned-reviews-v3/added_tokens.json',
 'bart-finetuned-reviews-v3/tokenizer.json')

In [11]:
# summarize function for summarization model
# max_input=1024, min_output=30, max_output=130
def summarize(text, max_input=1024, max_output=130, min_output=30):
    inputs = tokenizer(
        text,
        return_tensors='pt',
        max_length=max_input,
        truncation=True,
        padding=True
    ).to('cpu')

    # helps save memory during inference
    with torch.no_grad():
        summary_ids = model.generate(
            inputs['input_ids'],
            attention_mask=inputs['attention_mask'],
            max_length=max_output,
            min_length=min_output,
            length_penalty=2.0,
            num_beams=2,
            early_stopping=True
        )

    return tokenizer.decode(summary_ids[0], skip_special_tokens=True)

# function to summarize reviews
# combine up to max_reviews=10 and truncate to max_input_tokens=1024
def summarize_reviews(reviews, max_reviews=10, max_input_tokens=1024):
    if reviews.empty:
        return None
    
    combined = ' | '.join(reviews['text'].dropna().tolist()[:max_reviews])
    truncated = ' '.join(combined.split()[:max_input_tokens])

    return summarize(truncated)

# function to summarize descriptions
# combine all description bullet points and truncate to max_input_tokens=1024
def summarize_description(description: list, max_input_tokens=1024):
    if not description:
        return None
    
    combined = ' '.join(description)
    truncated = ' '.join(combined.split()[:max_input_tokens])

    return summarize(truncated, max_output=100, min_output=30)

In [27]:
# load and evaluate finetuned model v1 generations
# trained on summary (titles) from reviews.csv
# sampled 20000, 2 epochs
# Basically repeats the title of the review
tokenizer = AutoTokenizer.from_pretrained('../models/bart-finetuned-reviews')
model = AutoModelForSeq2SeqLM.from_pretrained('../models/bart-finetuned-reviews')
collator = DataCollatorForSeq2Seq(tokenizer, model=model, padding=True)

args = Seq2SeqTrainingArguments(
    output_dir='../models/bart-finetuned-reviews',
    predict_with_generate=True,
    fp16=torch.cuda.is_available()
)

trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    tokenizer=tokenizer,
    data_collator=collator,
    compute_metrics=compute_rouge
)

test_results = trainer.evaluate(tokenized_test)
print(test_results)

for i in range(3):
    sample = test_dataset[i]['text']
    print(f'Review: {sample}')
    print(f'Reference: {test['summary'].iloc[i]}')
    print(f'Generated: {summarize(sample)}\n')

/home/dangtrxn/cs4742_nlp/AmazonReview_Project/Amazon_Review/lib/python3.12/site-packages/transformers/models/bart/configuration_bart.py:176: UserWarning: Please make sure the config includes `forced_bos_token_id=0` in future versions. The config can simply be saved and uploaded again to be fixed.
  warnings.warn(
/tmp/ipykernel_345489/2601697081.py:15: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(
/home/dangtrxn/cs4742_nlp/AmazonReview_Project/Amazon_Review/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


{'eval_loss': 0.14575470983982086, 'eval_model_preparation_time': 0.0021, 'eval_rouge1': 5.1938, 'eval_rouge2': 0.3986, 'eval_rougeL': 4.4353, 'eval_rougeLsum': 4.4313, 'eval_runtime': 392.3226, 'eval_samples_per_second': 0.324, 'eval_steps_per_second': 0.041}
Review: We make up a coffee creamer with 1/4 part commercial coffee creamer, vanilla, butter creme, almond and Bavarian Creme flavors and the rest milk.  It tastes wonderful and is half the price of store-bought!<br /><br />It is excellent!
Reference: Homemade coffee creamer blend using multiple flavors turns out wonderful at half the store price
Generated: Great Coffee Creamer - Half the Price of Store-Bought!  Tastes great!  Half the price of store-bought!

Review: I love this candy.  After weight watchers I had to cut back but still have a craving for it.
Reference: Loved candy but reduced consumption due to dieting
Generated: Love this candy.  After weight watchers I had to cut back but still have a craving for it.  I love it

In [28]:
# load and evaluate finetuned model v2 generations
# trained on summary from refined_summaries.csv
# used entire dataset, 3 epochs
# best model, no overfitting
tokenizer = AutoTokenizer.from_pretrained('../models/bart-finetuned-reviews-v2')
model = AutoModelForSeq2SeqLM.from_pretrained('../models/bart-finetuned-reviews-v2')
collator = DataCollatorForSeq2Seq(tokenizer, model=model, padding=True)

args = Seq2SeqTrainingArguments(
    output_dir='../models/bart-finetuned-reviews-v2',
    predict_with_generate=True,
    fp16=torch.cuda.is_available()
)

trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    tokenizer=tokenizer,
    data_collator=collator,
    compute_metrics=compute_rouge
)

test_results = trainer.evaluate(tokenized_test)
print(test_results)

for i in range(3):
    sample = test_dataset[i]['text']
    print(f'Review: {sample}')
    print(f'Reference: {test['summary'].iloc[i]}')
    print(f'Generated: {summarize(sample)}\n')

/tmp/ipykernel_345489/1429922308.py:15: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


{'eval_loss': 0.09026647359132767, 'eval_model_preparation_time': 0.0026, 'eval_rouge1': 8.4424, 'eval_rouge2': 1.1089, 'eval_rougeL': 7.2716, 'eval_rougeLsum': 7.2579, 'eval_runtime': 404.4244, 'eval_samples_per_second': 0.314, 'eval_steps_per_second': 0.04}
Review: We make up a coffee creamer with 1/4 part commercial coffee creamer, vanilla, butter creme, almond and Bavarian Creme flavors and the rest milk.  It tastes wonderful and is half the price of store-bought!<br /><br />It is excellent!
Reference: Homemade coffee creamer blend using multiple flavors turns out wonderful at half the store price
Generated: Coffee creamer made with 1/4 commercial creamer, vanilla, butter creme, almond, and Bavarian creme flavors, and half milk; tastes wonderful and half the price of store-bought

Review: I love this candy.  After weight watchers I had to cut back but still have a craving for it.
Reference: Loved candy but reduced consumption due to dieting
Generated: Loves this candy and cut back 

In [29]:
# load and evaluate finetuned model v3 generations
# trained on summary from refined_summaries.csv
# used entire dataset, 3 epochs
# model overfitting
tokenizer = AutoTokenizer.from_pretrained('../models/bart-finetuned-reviews-v3')
model = AutoModelForSeq2SeqLM.from_pretrained('../models/bart-finetuned-reviews-v3')
collator = DataCollatorForSeq2Seq(tokenizer, model=model, padding=True)

args = Seq2SeqTrainingArguments(
    output_dir='../models/bart-finetuned-reviews-v3',
    predict_with_generate=True,
    fp16=torch.cuda.is_available()
)

trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    tokenizer=tokenizer,
    data_collator=collator,
    compute_metrics=compute_rouge
)

test_results = trainer.evaluate(tokenized_test)
print(test_results)

for i in range(3):
    sample = test_dataset[i]['text']
    print(f'Review: {sample}')
    print(f'Reference: {test['summary'].iloc[i]}')
    print(f'Generated: {summarize(sample)}\n')

/tmp/ipykernel_345489/1599000807.py:15: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


{'eval_loss': 0.08936727046966553, 'eval_model_preparation_time': 0.0025, 'eval_rouge1': 4.0261, 'eval_rouge2': 0.7398, 'eval_rougeL': 3.5004, 'eval_rougeLsum': 3.5312, 'eval_runtime': 641.2324, 'eval_samples_per_second': 0.198, 'eval_steps_per_second': 0.025}
Review: We make up a coffee creamer with 1/4 part commercial coffee creamer, vanilla, butter creme, almond and Bavarian Creme flavors and the rest milk.  It tastes wonderful and is half the price of store-bought!<br /><br />It is excellent!
Reference: Homemade coffee creamer blend using multiple flavors turns out wonderful at half the store price
Generated: Excellent coffee creamer made with 1/4 commercial coffee creaming, vanilla, butter creme, almond, and Bavarian creme flavors, and half milk; half the price of store-bought

Review: I love this candy.  After weight watchers I had to cut back but still have a craving for it.
Reference: Loved candy but reduced consumption due to dieting
Generated: Loves this candy after cutting b